# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LiquidMercury-tech/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I keep the feature vector simple and leakage-safe: demand, position, CTR, engagement, content age, and content metadata. I do not include raw identifiers or label-derived columns. Missing numeric values are filled with medians; categorical values get a safe missing bucket.

In [1]:
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
df = df.copy()
df['target_proxy'] = ((df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)) | ((df['sessions_90d'] >= 30) & ((df['engagement_rate'] < 30) | (df['scroll_rate'] < 30)))
features = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days', 'competition_level', 'content_type', 'main_intent', 'freshness_tier', 'position_tier']
X = df[features].copy()
for col in ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days']:
    X[col] = X[col].fillna(X[col].median())
for col in ['competition_level', 'content_type', 'main_intent', 'freshness_tier', 'position_tier']:
    X[col] = X[col].fillna('missing')
X = pd.get_dummies(X, columns=['competition_level', 'content_type', 'main_intent', 'freshness_tier', 'position_tier'], dummy_na=False)
print(f'Feature matrix rows: {X.shape[0]} ; columns: {X.shape[1]}')
print(X.head(2).to_string(index=False))


Feature matrix rows: 30000 ; columns: 34
 search_volume  competition  cpc  word_count  char_count  impressions_90d  clicks_90d  sessions_90d  ctr  avg_position  engagement_rate  scroll_rate  content_age_days  competition_level_HIGH  competition_level_LOW  competition_level_MEDIUM  competition_level_missing  content_type_comparison article  content_type_feedly article  content_type_keyword article  main_intent_commercial  main_intent_informational  main_intent_missing  main_intent_navigational  main_intent_transactional  freshness_tier_0-30  freshness_tier_181+  freshness_tier_31-90  freshness_tier_91-180  position_tier_deep  position_tier_page_1  position_tier_page_3_5  position_tier_striking  position_tier_top_3
          10.0         0.67 2.05      3221.0     20457.0             3803          29            17 0.76          10.6             5.88         4.55               187                    True                  False                     False                      False           

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Features are all measured within the same 90-day observation window before the review decision. Numeric metrics like CTR and engagement are taken from observed search behavior. Categorical fields like content type and intent are safe metadata that existed before the editor acted. I drop label-derived fields such as `trend_direction` because the same rule used to construct the label should not also be used as an input feature.

In [2]:
import pandas as pd
df = pd.read_csv(find_data_path())
notes = [('search_volume', 'demand proxy; median fill; available pre-decision'), ('ctr', 'click-through efficiency; median fill; observed before review'), ('avg_position', 'search ranking position; zero means no data; filtered where appropriate'), ('engagement_rate', 'on-page engagement; median fill; measured before review'), ('content_type', 'content metadata; missing bucket; available pre-decision'), ('trend_direction', 'label-derived; excluded to avoid leakage')]
for item in notes:
    print(f'{item[0]} -> {item[1]}')


search_volume -> demand proxy; median fill; available pre-decision
ctr -> click-through efficiency; median fill; observed before review
avg_position -> search ranking position; zero means no data; filtered where appropriate
engagement_rate -> on-page engagement; median fill; measured before review
content_type -> content metadata; missing bucket; available pre-decision
trend_direction -> label-derived; excluded to avoid leakage


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I test the strongest leakage risks directly. The main ones are label-derived columns and fields that would only be known after the review decision. A field like `trend_direction` is derived from `trend_pct`, which is also the basis of a label; using either one inside the feature set would make the model look strong while really re-learning the target definition. This notebook checks that the candidate feature set excludes those columns and only uses signals measured before the decision point.

In [3]:
import pandas as pd
df = pd.read_csv(find_data_path())
label_like = ['trend_direction', 'trend_pct', 'is_declining_label'] if 'is_declining_label' in df.columns else ['trend_direction', 'trend_pct']
available = [c for c in df.columns if c not in label_like and 'flag' not in c.lower() and 'score' not in c.lower()]
print('Label-derived fields to exclude:', label_like)
print('Safe candidate features:', available[:12])
print(f'Any product flags in starter data? {sum('flag' in c.lower() for c in df.columns)}')
print(f'Rows with avg_position=0: {int((df['avg_position'] == 0).sum())}; these are not true zero-rank rows and are filtered in rank-based scoring.')


Label-derived fields to exclude: ['trend_direction', 'trend_pct']
Safe candidate features: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used']
Any product flags in starter data? 0
Rows with avg_position=0: 1205; these are not true zero-rank rows and are filtered in rank-based scoring.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `trend_direction`: derived from the same metric used to define decline; would leak the label.
- `trend_pct`: same reason.
- `provider_used`, `model_used`: identifiers or implementation metadata, not outcome evidence.
- `content_id`, `client_id`: they are grouping keys, not model features.
- any product flag or score column: not in the starter dataset; if rebuilt, it would violate the observable-only principle.

In [4]:
excluded = ['trend_direction', 'trend_pct', 'provider_used', 'model_used', 'content_id', 'client_id']
print('Excluded from usable feature set:')
for x in excluded:
    print('-', x)


Excluded from usable feature set:
- trend_direction
- trend_pct
- provider_used
- model_used
- content_id
- client_id


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.